# Using a Language Model

Everything so far we trained ourselves. This time we **download a model
someone else already trained** and use it.

Steps:
1. Load a model from Hugging Face
2. Ask it something
3. See how the system prompt changes its behaviour
4. See how the generation settings change its answers
5. Zero-shot vs few-shot prompting

Model: `Qwen2.5-0.5B-Instruct` - 0.5 billion parameters, about 1 GB.
Runs on CPU. A GPU makes it faster but is not required.

In [ ]:
# transformers is the library for downloading and running these models.
!pip install -q -U transformers

In [ ]:
import torch
from transformers import pipeline

# Use the GPU if there is one, otherwise fall back to CPU.
DEVICE = 0 if torch.cuda.is_available() else -1
print('Running on:', 'GPU' if DEVICE == 0 else 'CPU')

## 1. Load the model

`pipeline` is the easy way in. One line downloads the model and gets it ready.

First run downloads ~1 GB. After that it is cached.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

chat = pipeline('text-generation', model=MODEL_ID, device=DEVICE)

print(f'{chat.model.num_parameters()/1e6:.0f}M parameters')

## 2. Ask it something

Messages go in as a list of `{role, content}`. The pipeline handles the rest.

In [ ]:
def ask(question, system='You are a helpful assistant.', **settings):
    """Send a question to the model and return just its reply text."""
    out = chat(
        [{'role': 'system', 'content': system},
         {'role': 'user',   'content': question}],
        max_new_tokens=settings.pop('max_new_tokens', 120),
        **settings,
    )
    # The pipeline returns the whole conversation - the last message is the answer.
    return out[0]['generated_text'][-1]['content']

In [ ]:
# On CPU this takes a few seconds. Be patient.
print(ask('What is a transformer in machine learning? Answer in two sentences.'))

### What the pipeline is doing for us

Two things happen behind that one call, and they are worth seeing once.

**First - the messages become a single string** with special markers, because
that is the format the model was actually trained on.

In [ ]:
messages = [{'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'user',   'content': 'Hello!'}]

# Every model family uses different markers. This one is Qwen's.
print(chat.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

**Second - that string becomes numbers.** The model never sees text, only token ids.

In [ ]:
text = 'Fine-tuning is unbelievably useful'
ids  = chat.tokenizer.encode(text)

# Notice that long or rare words get split into several pieces.
print('ids   :', ids)
print('tokens:', [chat.tokenizer.decode([i]) for i in ids])

## 3. The system prompt changes the behaviour

Same question. Different instructions. No retraining.

In [ ]:
question = 'Why is my code slow?'

for system in ['You are a helpful assistant.',
               'You are a grumpy senior engineer. Answer in one short line.',
               'You explain things to a 10 year old. Use a simple example.']:
    print('SYSTEM:', system)
    print('REPLY :', ask(question, system=system, max_new_tokens=60))
    print('-' * 70)

## 4. Generation settings

The model does not pick one answer - it produces a probability for every
possible next token. How we choose from those probabilities is up to us.

- `do_sample=False` - always take the most likely token. Same answer every time.
- `do_sample=True` + `temperature` - pick randomly. Higher = more surprising.

In [ ]:
prompt = 'Give me a name for a coffee shop run by programmers.'

print('GREEDY (do_sample=False) - run it twice, identical both times:')
for _ in range(2):
    print(' ', ask(prompt, do_sample=False, max_new_tokens=20))

print('\nSAMPLING (temperature=1.2) - different every time:')
for _ in range(3):
    print(' ', ask(prompt, do_sample=True, temperature=1.2, top_p=0.95, max_new_tokens=20))

**Rule of thumb:** `do_sample=False` when you want a reliable answer
(extraction, classification). Sampling when you want variety (brainstorming, writing).

## 5. Zero-shot vs few-shot

**Zero-shot** = just ask. **Few-shot** = show a couple of examples first.

In [ ]:
message = 'hi team, the login page throws a 500 since the deploy, QA is blocked - Priya'

# Zero-shot: we describe what we want, but show nothing.
zero_shot = f'Extract client, urgency and area from this message as JSON:\n{message}'

print(ask(zero_shot, do_sample=False))

In [ ]:
# Few-shot: same task, but we show two worked examples first.
few_shot = f"""Extract client, urgency and area as JSON.

Message: checkout is failing for everyone, we are losing sales - Sam, Karma Foods
JSON: {{"client": "Karma Foods", "urgency": "critical", "area": "payments"}}

Message: could we get dark mode sometime? no rush - Leena, Vertex Labs
JSON: {{"client": "Vertex Labs", "urgency": "low", "area": "ui"}}

Message: {message}
JSON:"""

print(ask(few_shot, do_sample=False))

The few-shot version should stick to the format much better.

But notice it is still not perfect, and we just spent a lot of tokens on examples.
**When prompting is not enough, the next step is fine-tuning** - that is next week.

| Problem | Fix |
|---|---|
| Model doesn't know your facts | RAG |
| Model gets tone or format wrong | Fine-tuning |
| Model just needs clearer instructions | Prompting (today) |

## 6. Your turn

Play with the model here.

In [ ]:
print(ask('Write a git commit message for: fixed a null check in the login handler',
          system='You write short conventional-commit messages. Reply with one line only.',
          do_sample=False, max_new_tokens=30))

## Assignment

**Load a different model from Hugging Face and build something small with it.**

Go to [huggingface.co/models](https://huggingface.co/models) and pick one. Good small options:

- `HuggingFaceTB/SmolLM2-360M-Instruct` - smaller and faster than today's
- `Qwen/Qwen2.5-1.5B-Instruct` - bigger, noticeably better, slower
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

Then pick one of these and make it work:

1. **Tone rewriter** - turn a blunt message into a polite client email
2. **Commit message writer** - describe a change, get a conventional commit line
3. **Meeting note summariser** - paste messy notes, get 3 bullet points
4. **Standup bot** - given what you did, format it as yesterday / today / blockers

For whichever you pick, answer these in a markdown cell:

- What system prompt did you land on, and what did you try first?
- Did few-shot examples help? Show a before and after.
- Where does it still fail? Be specific - that failure is what fine-tuning fixes.

**Also try a pipeline for a different task.** `text-generation` is only one kind:

```python
sentiment = pipeline('sentiment-analysis')
print(sentiment('This lab was surprisingly fun'))
```

Others to try: `summarization`, `translation`, `fill-mask`, `ner`.